# 第7回実習：日本語AI推論＋UR5e／Robotiq 2F-85によるPick & Place統合

**研究開発途中の初期統合試験。** 第6回実習で確認した円柱の把持・搬送・配置を、学習済みSentence Transformerによる日本語指示のタスク選択と連携する。

- 既存の01～06ノートブックは変更せず、この07だけを実行する。
- **AIが選択するのは作業IDのみ**。試料の位置、把持条件、関節目標、動作系列はあらかじめ決めてある。
- 現時点の物体は**出発側（緑の台）にある円柱1個のみ**。対応する自動搬送は**緑の出発台 → 青の搬送先台（+Y方向100 mm）**のみ。逆方向を指示されても動作しない。
- 推論を単独で検証し、安全側のゲートを通過した場合のみ物理シミュレーションを開始する。REJECTでは新規目標を変更しない。
- 第6回と同じ約24秒の搬送を実行した後、REJECT保留・HOME復帰を加える試験であり、未知指示に対するAI認識精度や実機安全性を保証しない。

**実行の目安：** Colab CPU（無料ランタイム）でセル1から順番に実行。**セル5の事前判定結果を確認してからセル6の動画生成**へ進む。

## セル1：Colab環境の準備（第6回＋学習済み日本語モデル）

In [ ]:
# セル1：Colabの依存関係（CPU環境、OSMesa使用）
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk > /dev/null
%pip -q install "mujoco>=3.3,<4" "mujoco-menagerie==2026.10.3" "imageio[ffmpeg]" numpy pandas pillow matplotlib scipy "sentence-transformers>=3.0"
print('セル1完了。第7回セル1完了。次はセル2を実行してください。')

## セル2：ロボット・グリッパー・試料・2つの支持台

第6回で使用したMuJoCoモデル設定値をそのまま引き継ぐ。円柱は**freejointを持つ自由物体**で、グリッパーと溶接・固定していない。

In [ ]:
# セル2：MuJoCoモデル構築（公式Menagerieモデルを使用）
import os
os.environ['MUJOCO_GL'] = 'osmesa'  # mujoco import より先

import mujoco
import mujoco_menagerie as mm
import numpy as np
import pandas as pd
from pathlib import Path

np.set_printoptions(precision=4, suppress=True)
HOME_Q = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0.0])
ARM_JOINTS = ['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint',
              'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint']
ARM_ACTUATORS = ['shoulder_pan', 'shoulder_lift', 'elbow',
                 'wrist_1', 'wrist_2', 'wrist_3']

RADIUS_M = 0.016     # 直径32mm
HALF_HEIGHT_M = 0.028 # 高さ56mm
MASS_KG = 0.030      # 30g：探索用の値
LIFT_M = 0.075       # 75mm
DEST_DELTA_XY_M = np.array([0.0, 0.10])  # 初期搬送先：+Y方向に100mm
PEDESTAL_HALF_H = 0.035
PEDESTAL_R = 0.011

# 毎回別のMjSpecを作成する。コンパイル済みspecに追加する際のバージョン依存を回避。
def make_spec():
    ur = mm.get('universal_robots_ur5e').spec('scene')
    grip = mm.get('robotiq_2f85').spec('2f85')
    mount = ur.site('attachment_site')
    if mount is None:
        raise RuntimeError('公式UR5eに attachment_site が見つかりません')
    ur.attach(grip, prefix='grip_', site=mount)
    return ur

def get_id(m, kind, name):
    k = mujoco.mj_name2id(m, kind, name)
    if k < 0:
        raise RuntimeError(f'モデル内に {name} がありません')
    return int(k)

def set_home(m, d):
    joint_ids = [get_id(m, mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
    actuator_ids = [get_id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]
    mujoco.mj_resetData(m, d)
    for jid, angle in zip(joint_ids, HOME_Q):
        d.qpos[int(m.jnt_qposadr[jid])] = float(angle)
    for aid, angle in zip(actuator_ids, HOME_Q):
        d.ctrl[aid] = float(angle)
    gaid = get_id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, 'grip_fingers_actuator')
    d.ctrl[gaid] = 0.0
    mujoco.mj_forward(m, d)

probe_model = make_spec().compile()
probe_data = mujoco.MjData(probe_model)
set_home(probe_model, probe_data)
probe_pinch_id = get_id(probe_model, mujoco.mjtObj.mjOBJ_SITE, 'grip_pinch')
PINCH_HOME = np.array(probe_data.site_xpos[probe_pinch_id], dtype=float).copy()
PINCH_ROT_HOME = np.array(probe_data.site_xmat[probe_pinch_id]).reshape(3,3).copy()
axis_dot = abs(float(PINCH_ROT_HOME[:,2] @ np.array([0., 0., 1.])))
print('HOMEでの把持点 [m]:', PINCH_HOME, '／把持軸と鉛直軸の平行度:', round(axis_dot,3))
if axis_dot < 0.75:
    raise RuntimeError('HOMEで把持軸が大きく傾いています。円柱の向き・置き方を再検討してください。')
if PINCH_HOME[2] < HALF_HEIGHT_M + 2*PEDESTAL_HALF_H + 0.025:
    raise RuntimeError('試料を置く高さが足りません。HOME姿勢を調整してください。')

# 支持台上面は試料底面よりわずかに低くする（重力で試料が接地する）
SUPPORT_TOP_Z = float(PINCH_HOME[2] - HALF_HEIGHT_M - 0.0005)
PILLAR_CENTER_Z = SUPPORT_TOP_Z - PEDESTAL_HALF_H
DESTINATION_XY = PINCH_HOME[:2] + DEST_DELTA_XY_M

spec = make_spec()
spec.worldbody.add_geom(
    name='sample_pedestal', type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    pos=[float(PINCH_HOME[0]), float(PINCH_HOME[1]), PILLAR_CENTER_Z],
    size=[PEDESTAL_R, PEDESTAL_HALF_H, 0.0],
    rgba=[0.23, 0.67, 0.60, 1.0],
    friction=[1.1, 0.005, 0.0001],
)
# 初期配置の支持台と同一高さの搬送先支持台を追加
spec.worldbody.add_geom(
    name='destination_pedestal', type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    pos=[float(DESTINATION_XY[0]), float(DESTINATION_XY[1]), PILLAR_CENTER_Z],
    size=[PEDESTAL_R, PEDESTAL_HALF_H, 0.0],
    rgba=[0.28, 0.50, 0.95, 1.0],  # 青色：搬送先
    friction=[1.1, 0.005, 0.0001],
)
object_body = spec.worldbody.add_body(name='test_sample', pos=PINCH_HOME.tolist())
object_body.add_freejoint()
object_body.add_geom(
    name='test_sample_cylinder', type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    size=[RADIUS_M, HALF_HEIGHT_M, 0.0],
    mass=MASS_KG, friction=[1.5, 0.01, 0.001],
    rgba=[0.96, 0.56, 0.17, 1.0],
)
model = spec.compile()
data = mujoco.MjData(model)
set_home(model, data)

arm_joint_ids = [get_id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
arm_qpos_ids = np.array([int(model.jnt_qposadr[j]) for j in arm_joint_ids])
arm_dof_ids = np.array([int(model.jnt_dofadr[j]) for j in arm_joint_ids])
arm_actuator_ids = [get_id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]
fr_id = get_id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, 'grip_fingers_actuator')
left_j = get_id(model, mujoco.mjtObj.mjOBJ_JOINT, 'grip_left_driver_joint')
right_j = get_id(model, mujoco.mjtObj.mjOBJ_JOINT, 'grip_right_driver_joint')
left_qpos_id = int(model.jnt_qposadr[left_j]); right_qpos_id = int(model.jnt_qposadr[right_j])
pinch_id = get_id(model, mujoco.mjtObj.mjOBJ_SITE, 'grip_pinch')
obj_geom = get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'test_sample_cylinder')
source_support_geom = get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'sample_pedestal')
target_support_geom = get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'destination_pedestal')
obj_body_id = get_id(model, mujoco.mjtObj.mjOBJ_BODY, 'test_sample')
obj_j = model.body_jntadr[obj_body_id]
obj_qpos_id = int(model.jnt_qposadr[obj_j])
left_pads = {get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'grip_'+n)
             for n in ['left_pad1','left_pad2']}
right_pads = {get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'grip_'+n)
              for n in ['right_pad1','right_pad2']}

assert model.nu == 7, f'想定外のアクチュエータ数: {model.nu}'
assert model.jnt_type[obj_j] == mujoco.mjtJoint.mjJNT_FREE, '試料は自由物体である必要があります'
assert abs(float(data.qpos[obj_qpos_id+2])-PINCH_HOME[2]) < 0.001
assert np.allclose(data.site_xpos[pinch_id], PINCH_HOME, atol=0.001)
print('モデル構築成功:', 'nq=', model.nq, 'nu=', model.nu)
print('円柱:', f'直径{RADIUS_M*2000:.0f}mm／高さ{HALF_HEIGHT_M*2000:.0f}mm／質量{MASS_KG*1000:.0f}g')
print('出発台と搬送先台の上面 z [m]:', round(SUPPORT_TOP_Z,4))
print('初期試料XY [m]:',np.round(PINCH_HOME[:2],4))
print('目標の支持台XY [m]:',np.round(DESTINATION_XY,4),'／水平移動量 [mm]:',round(np.linalg.norm(DEST_DELTA_XY_M)*1000,1))
print('グリッパー制御範囲:', model.actuator_ctrlrange[fr_id])
print('次はセル3で緑色の出発台・青色の搬送先台・オレンジ色の円柱の位置を確認してください。')

## セル3：初期配置を確認

**オレンジ：円柱／緑：出発台／青：搬送先台**。静止画像で配置を確認する。この時点での左右パッド接触Falseは正常。

In [ ]:
# セル3：2視点で試料とグリッパーを表示
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
import imageio.v2 as imageio

WIDTH, HEIGHT = 480, 360
renderer = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)
whole_cam = mujoco.MjvCamera()
whole_cam.type = mujoco.mjtCamera.mjCAMERA_FREE
whole_cam.azimuth = 135
whole_cam.elevation = -24
whole_cam.distance = 1.7
whole_cam.lookat[:] = [float(PINCH_HOME[0]+DEST_DELTA_XY_M[0]/2), float(PINCH_HOME[1]+DEST_DELTA_XY_M[1]/2), .35]
near_cam = mujoco.MjvCamera()
near_cam.type = mujoco.mjtCamera.mjCAMERA_FREE
near_cam.azimuth = 135
near_cam.elevation = -18
near_cam.distance = 0.36
fontpath = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font = ImageFont.truetype(fontpath, 18) if Path(fontpath).exists() else ImageFont.load_default()
smallfont = ImageFont.truetype(fontpath, 14) if Path(fontpath).exists() else ImageFont.load_default()

def render_views(phase, cmd, left_touch=False, right_touch=False):
    # ※ mujoco.mj_forward()は物理時間を進めず、現時点の運動学を更新する
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=whole_cam)
    image1 = Image.fromarray(renderer.render().copy())
    near_cam.lookat[:] = data.site_xpos[pinch_id]
    renderer.update_scene(data, camera=near_cam)
    image2 = Image.fromarray(renderer.render().copy())
    canvas = Image.new('RGB', (2*WIDTH, HEIGHT+86), (15,27,42))
    canvas.paste(image1, (0,62)); canvas.paste(image2, (WIDTH,62))
    dr = ImageDraw.Draw(canvas)
    dr.text((12,5),f'UR5e + Robotiq 2F-85 | Pick & Place | {phase}',font=font,fill='white')
    dr.text((12,36),'全体 (左)',font=smallfont,fill=(182,215,237))
    dr.text((WIDTH+12,36),'指先拡大 (右) | 開閉: '+str(int(cmd))+'/255',font=smallfont,fill=(182,215,237))
    dr.text((12,HEIGHT+68),f'左右接触={left_touch}/{right_touch}   緑=出発台, 青=搬送先台   ※試料は自由物体',
            font=smallfont,fill=(248,213,166))
    return np.array(canvas)

display(Image.fromarray(render_views('INITIAL',0)))
print('確認：オレンジの試料は緑の出発台上にあり、青の搬送先台が別位置にありますか？')

## セル4：搬送に必要な目標関節姿勢（IK）

第6回と同じ、約75 mmのLIFT、約100 mmのTRANSFER、搬送先でのPLACE姿勢を計算する。IKの収束は経路全体の衝突安全性を保証しない。

In [ ]:
# セル4：3段階の逆運動学（前の解を次の初期値に使用）
from scipy.spatial.transform import Rotation

ik_data = mujoco.MjData(model)
set_home(model, ik_data)
start_pos = np.array(ik_data.site_xpos[pinch_id]).copy()
start_rot = np.array(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
q_min = np.array([model.actuator_ctrlrange[a,0] for a in arm_actuator_ids])
q_max = np.array([model.actuator_ctrlrange[a,1] for a in arm_actuator_ids])

def solve_goal_ik(pos_target, seed_q, name):
    q = np.array(seed_q, dtype=float).copy()
    for it in range(180):
        ik_data.qpos[arm_qpos_ids] = q
        mujoco.mj_forward(model, ik_data)
        pos = np.asarray(ik_data.site_xpos[pinch_id]).copy()
        R = np.asarray(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
        pe = np.asarray(pos_target)-pos
        re = Rotation.from_matrix(start_rot @ R.T).as_rotvec()
        if np.linalg.norm(pe) < 0.0025 and np.linalg.norm(re) < 0.09:
            break
        jp = np.zeros((3, model.nv)); jr = np.zeros((3, model.nv))
        mujoco.mj_jacSite(model, ik_data, jp, jr, pinch_id)
        J = np.vstack([jp[:,arm_dof_ids], 0.3*jr[:,arm_dof_ids]])
        err = np.r_[pe, 0.3*re]
        damp = .02
        dq = J.T @ np.linalg.solve(J@J.T+damp**2*np.eye(6),err)
        q = np.clip(q+np.clip(dq,-.09,.09),q_min,q_max)
    ik_data.qpos[arm_qpos_ids] = q
    mujoco.mj_forward(model, ik_data)
    final_p=np.array(ik_data.site_xpos[pinch_id]).copy()
    final_R=np.array(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
    pos_err=np.linalg.norm(final_p-pos_target)
    rot_err=np.linalg.norm(Rotation.from_matrix(start_rot@final_R.T).as_rotvec())
    print(f'{name}: 位置誤差={pos_err*1000:.2f}mm / 姿勢誤差={rot_err:.4f}rad / 反復数={it+1}')
    if pos_err > 0.008 or rot_err > 0.12:
        raise RuntimeError(f'{name} のIK精度が不足。搬送距離を短くするなど調整してください。')
    return q

LIFT_TARGET = start_pos + np.array([0.,0.,LIFT_M])
TRANSFER_TARGET = LIFT_TARGET + np.r_[DEST_DELTA_XY_M, 0.0]
PLACE_TARGET = start_pos + np.r_[DEST_DELTA_XY_M, 0.0]
LIFT_Q = solve_goal_ik(LIFT_TARGET,HOME_Q,'LIFT')
TRANSFER_Q = solve_goal_ik(TRANSFER_TARGET,LIFT_Q,'TRANSFER')
PLACE_Q = solve_goal_ik(PLACE_TARGET,TRANSFER_Q,'PLACE')
print('目標関節姿勢 LIFT:',np.round(LIFT_Q,3))
print('目標関節姿勢 TRANSFER:',np.round(TRANSFER_Q,3))
print('目標関節姿勢 PLACE:',np.round(PLACE_Q,3))
print('IK計算完了。次はセル5の接触物理シミュレーションです。')

## セル5：日本語作業指示のAI判定と実行許可ゲート（**ここで一度確認**）

学習済みの `paraphrase-multilingual-MiniLM-L12-v2` を読み込み、**同じ作業を言い換えた表現**に対する類似度でタスク候補を選ぶ。学習済みモデル本体は再学習しない。

運動シーンで許可するのは `MOVE_LEFT_TO_RIGHT` の1種類。`HOME` は復帰のみ、`REJECT` は指令保留とする。

**注意**：前回04で使った「左右を同時に含むと棄却」は、今回の「左から右へ運ぶ」という正当な指示を誤って棄却するため、再利用しない。代わりに**出発物と搬送先を明示する規則**と、対応可能な搬送方向を組み合わせる。これは確実な意味理解ではなく、限定的なルールである。

In [ ]:
# セル5：Sentence Transformer＋対象物／方向／安全ゲート
import time
import re
from sentence_transformers import SentenceTransformer

MODEL_ID = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
text_model = SentenceTransformer(MODEL_ID, device='cpu')
# 04の推論方式（正規化した埋め込み＋コサイン類似度）を継承し、
# 07の物体搬送に合う作業定義へ更新する。
TASK_EXAMPLES = {
    'MOVE_LEFT_TO_RIGHT': '左の円柱を右の青い台へ移動して',
    'UNSUPPORTED_REVERSE': '右の円柱を左の緑の台へ移動して',
    'HOME': 'ロボットアームを待機姿勢に戻して',
}
TASK_LABELS = list(TASK_EXAMPLES)
ref_vectors = text_model.encode([TASK_EXAMPLES[k] for k in TASK_LABELS], normalize_embeddings=True)
MIN_SCORE = 0.60
MIN_MARGIN = 0.025

def lexical_direction_guard(instruction):
    """物体・搬送元・搬送先の限定的な検査。自然言語一般の理解ではない。"""
    s = re.sub(r'\s+', '', instruction)
    if any(w in s for w in ['どちらでも', 'どちらか', '適当に', '任意', '左か右']):
        return False, '曖昧な対象／行き先'
    if any(w in s for w in ['右の円柱', '右側の円柱', '左の台', '左側の台', '緑の台へ', '緑の台に', '右から左']):
        return False, '現在のシーンで未対応の搬送方向'
    # 物体は緑側の円柱、搬送先は右側の青い台。
    src_ok = ('左' in s or '緑' in s) and ('円柱' in s or '試料' in s)
    dst_ok = ('右' in s or '青' in s) and ('台' in s or '置き場' in s)
    verb_ok = any(w in s for w in ['移動', '運ん', '搬送', '置い', '持って'])
    if not (src_ok and dst_ok and verb_ok):
        return False, '搬送元・搬送先・作業内容の明示不足'
    return True, '方向・作業条件が明示されている'

def infer_pick_place_task(instruction):
    tic = time.perf_counter()
    vec = text_model.encode([instruction], normalize_embeddings=True)[0]
    similarities = ref_vectors @ vec
    order = np.argsort(similarities)[::-1]
    ix0, ix1 = map(int, order[:2])
    proposed = TASK_LABELS[ix0]
    score = float(similarities[ix0]); margin = score - float(similarities[ix1])
    if score < MIN_SCORE or margin < MIN_MARGIN:
        decision, reason = 'REJECT', '意味的類似度または上位2候補の差が不足'
    elif proposed == 'HOME':
        s = re.sub(r'\s+', '', instruction)
        if any(w in s for w in ['戻', 'ホーム', '待機']) and not any(w in s for w in ['円柱', '運ん', '移動', '搬送']):
            decision, reason = 'HOME', 'HOMEのみ許可'
        else:
            decision, reason = 'REJECT', 'HOMEの指示が明確でない'
    elif proposed != 'MOVE_LEFT_TO_RIGHT':
        decision, reason = 'REJECT', '認識候補が未対応方向'
    else:
        ok, reason = lexical_direction_guard(instruction)
        decision = 'MOVE_LEFT_TO_RIGHT' if ok else 'REJECT'
    return {
        'instruction': instruction, 'proposed_task': proposed, 'task_id': decision,
        'score': round(score, 4), 'margin': round(margin, 4),
        'reason': reason, 'inference_seconds': round(time.perf_counter()-tic, 4),
        'scores': {t: round(float(similarities[j]), 4) for j,t in enumerate(TASK_LABELS)},
    }

# 実演用の選定済み例文。独立評価・未知言語への汎化性能の証拠ではない。
INSTRUCTIONS = [
    ('左の円柱を右の青い台へ移動して', 'MOVE_LEFT_TO_RIGHT'),
    ('左右どちらでもよいから適当に移動して', 'REJECT'),
    ('右の円柱を左の緑の台へ移動して', 'REJECT'),
    ('ロボットアームを待機姿勢に戻して', 'HOME'),
]
INFERENCE_RESULTS = [infer_pick_place_task(x) for x,_ in INSTRUCTIONS]
for result, (_, expected) in zip(INFERENCE_RESULTS, INSTRUCTIONS):
    print(f"入力: {result['instruction']}\n  推論={result['proposed_task']}, 判定={result['task_id']} (期待={expected}), score={result['score']:.4f}, margin={result['margin']:.4f}\n  根拠: {result['reason']}")
if any(r['task_id'] != expected for r,(_,expected) in zip(INFERENCE_RESULTS,INSTRUCTIONS)):
    raise RuntimeError('推論結果が事前の期待と一致しません。セル6は実行せず、判定結果を確認してください。')
if not (np.isclose(data.ctrl[fr_id], 0) and np.allclose(data.ctrl[arm_actuator_ids], HOME_Q)):
    raise RuntimeError('ロボットが初期状態でありません。セル2から実行し直してください。')
print('事前チェック通過：搬送1件、棄却2件、HOME1件。次のセル6へ進めます。')

## セル6：AIが許可した場合のみPick & Placeを実行してMP4とCSVを作成

第6回で動作確認した**13フェーズ、24秒**を基礎にする。試行の後、あいまいな指示に対する `REJECT_HOLD`（1.5秒）と、明確なHOME指示による `HOME`（2.5秒）を追加する。合計**約28秒（224フレーム、8fps）**。

**REJECT中は新しいアーム・グリッパー目標を発行せず**直前の指令値を保持するが、慣性や物理収束による微小な動きは残り得る。HOMEも事前の関節角制御であり、衝突回避軌道を自動生成していない。

In [ ]:
# セル5：接触・搬送・配置・退避をログ化（成功するとは限りません）
from IPython.display import display, Video

OUT_MP4 = 'ur5e_ai_pick_place_test.mp4'
OUT_CSV = 'ur5e_ai_pick_place_frames.csv'
OUT_SUMMARY = 'ur5e_ai_pick_place_summary.csv'
OUT_INFERENCE = 'ur5e_ai_pick_place_inference.csv'
# AI推論を本当に実行し、許可されたタスクだけシミュレーションへ渡す。
assert INFERENCE_RESULTS[0]['task_id'] == 'MOVE_LEFT_TO_RIGHT', 'AIが搬送指示を許可していません。動作は開始しません。'
assert INFERENCE_RESULTS[1]['task_id'] == 'REJECT', '曖昧な指示を拒否できません。動作は開始しません。'
assert INFERENCE_RESULTS[2]['task_id'] == 'REJECT', '逆方向の指示を拒否できません。動作は開始しません。'
assert INFERENCE_RESULTS[3]['task_id'] == 'HOME', 'HOMEを正しく判定できません。動作は開始しません。'
pd.DataFrame([{k:v for k,v in r.items() if k!='scores'} for r in INFERENCE_RESULTS]).to_csv(OUT_INFERENCE, index=False, encoding='utf-8-sig')
FPS = 8
# phase, 秒数, 目標アーム姿勢, グリッパー指令(0=開, 255=閉)
PHASES = [
    ('SETTLE_OPEN',     1.25, HOME_Q,       0.0),
    ('CLOSE',           3.00, HOME_Q,     255.0),
    ('GRASP_HOLD',      1.00, HOME_Q,     255.0),
    ('LIFT',            3.00, LIFT_Q,     255.0),
    ('HOLD_UP',         1.00, LIFT_Q,     255.0),
    ('TRANSFER',        3.00, TRANSFER_Q, 255.0),
    ('HOLD_AT_TARGET',  0.75, TRANSFER_Q, 255.0),
    ('LOWER',           3.00, PLACE_Q,    255.0),
    ('SETTLE_AT_TARGET',0.75, PLACE_Q,    255.0),
    ('RELEASE',         2.50, PLACE_Q,      0.0),
    ('WAIT_AFTER_RELEASE',1.25, PLACE_Q,   0.0),
    ('RETRACT',         2.50, TRANSFER_Q,  0.0),
    ('FINAL_HOLD',      1.00, TRANSFER_Q,  0.0),
]

# 24秒の物体操作のあとに、REJECTの指令保留とHOME復帰を追加
PHASES += [
    ('REJECT_HOLD', 1.50, TRANSFER_Q, 0.0),
    ('HOME',        2.50, HOME_Q,     0.0),
]

def inspect_contacts():
    left_count=right_count=source_count=target_count=0
    left_force=right_force=0.0
    for k in range(data.ncon):
        c=data.contact[k]
        a,b=int(c.geom1),int(c.geom2)
        if a != obj_geom and b != obj_geom:
            continue
        other=b if a == obj_geom else a
        if other in left_pads or other in right_pads:
            cf=np.zeros(6)
            mujoco.mj_contactForce(model,data,k,cf)
            if other in left_pads:
                left_count += 1
                left_force += max(0.0,float(cf[0]))
            if other in right_pads:
                right_count += 1
                right_force += max(0.0,float(cf[0]))
        if other == source_support_geom:
            source_count += 1
        if other == target_support_geom:
            target_count += 1
    return left_count,right_count,left_force,right_force,source_count,target_count

set_home(model,data)
first_object_pos=np.asarray(data.xpos[obj_body_id]).copy()
frame_count=0
frame_records=[]
phase_records=[]
h=float(model.opt.timestep)
if not (0<h<0.1):
    raise RuntimeError('MuJoCo時間刻みが想定外です')

with imageio.get_writer(OUT_MP4,fps=FPS,codec='libx264',quality=7,macro_block_size=1) as writer:
    for phase,duration,desired_q,grip_cmd in PHASES:
        phase_start_q=np.array([data.ctrl[a] for a in arm_actuator_ids])
        nframes=int(round(duration*FPS))
        for j in range(nframes):
            r=(j+1)/nframes
            s=r*r*(3-2*r)  # アーム姿勢の緩やかな開始・終了
            qcmd=(1-s)*phase_start_q+s*np.asarray(desired_q)
            if phase == 'REJECT_HOLD':
                # 事前の目標指令を変更しない。物理状態を人工的に固定することはしない。
                qcmd=np.array([data.ctrl[a] for a in arm_actuator_ids], dtype=float)
                grip_cmd=float(data.ctrl[fr_id])
            for aid,v in zip(arm_actuator_ids,qcmd):
                data.ctrl[aid]=float(v)
            data.ctrl[fr_id]=float(grip_cmd)
            desired_t=(frame_count+1)/FPS
            while data.time+h/2 < desired_t:
                mujoco.mj_step(model,data)
            if not (np.all(np.isfinite(data.qpos)) and np.all(np.isfinite(data.qvel))):
                raise RuntimeError(f'{phase}: 数値異常。動画とログを確認してください。')
            lc,rc,lf,rf,sc,tc=inspect_contacts()
            obj_p=np.asarray(data.xpos[obj_body_id]).copy()
            pinch_p=np.asarray(data.site_xpos[pinch_id]).copy()
            dest_err=np.linalg.norm(obj_p[:2]-DESTINATION_XY)
            src_err=np.linalg.norm(obj_p[:2]-PINCH_HOME[:2])
            rec={
                'time_s':round(float(data.time),4), 'phase':phase,
                'ai_task_id': ('REJECT' if phase=='REJECT_HOLD' else 'HOME' if phase=='HOME' else 'MOVE_LEFT_TO_RIGHT'),
                'gripper_ctrl_0_255':float(grip_cmd),
                'left_driver_rad':float(data.qpos[left_qpos_id]),
                'right_driver_rad':float(data.qpos[right_qpos_id]),
                'left_pad_contact_count':int(lc), 'right_pad_contact_count':int(rc),
                'left_pad_normal_force_N':round(lf,3),
                'right_pad_normal_force_N':round(rf,3),
                'both_pads_touch':bool(lc>0 and rc>0),
                'source_pedestal_contact_count':int(sc),
                'target_pedestal_contact_count':int(tc),
                'sample_x_m':float(obj_p[0]),'sample_y_m':float(obj_p[1]),'sample_z_m':float(obj_p[2]),
                'pinch_x_m':float(pinch_p[0]),'pinch_y_m':float(pinch_p[1]),'pinch_z_m':float(pinch_p[2]),
                'sample_lift_from_initial_m':float(obj_p[2]-first_object_pos[2]),
                'sample_clearance_above_support_m':float(obj_p[2]-HALF_HEIGHT_M-SUPPORT_TOP_Z),
                'sample_to_pinch_distance_m':float(np.linalg.norm(obj_p-pinch_p)),
                'sample_vertical_alignment':abs(float(np.asarray(data.xmat[obj_body_id]).reshape(3,3)[:,2] @ np.array([0.,0.,1.]))),
                'sample_to_target_xy_error_m':float(dest_err),
                'sample_to_source_xy_error_m':float(src_err),
                'max_arm_tracking_error_rad':float(np.max(np.abs(data.qpos[arm_qpos_ids]-qcmd))),
                'contacts_all_n':int(data.ncon),
            }
            frame_records.append(rec)
            writer.append_data(render_views('AI '+rec['ai_task_id']+' | '+phase,grip_cmd,lc>0,rc>0))
            frame_count+=1
        last=frame_records[-1]
        phase_records.append({
            'phase':phase,'end_time_s':last['time_s'],
            'end_sample_x_m':last['sample_x_m'],'end_sample_y_m':last['sample_y_m'],
            'end_sample_z_m':last['sample_z_m'],
            'end_target_xy_error_m':last['sample_to_target_xy_error_m'],
            'end_sample_support_clearance_m':last['sample_clearance_above_support_m'],
            'end_both_pads_touch':last['both_pads_touch'],
            'end_target_support_contact_count':last['target_pedestal_contact_count'],
            'end_arm_tracking_error_rad':last['max_arm_tracking_error_rad'],
        })

df=pd.DataFrame(frame_records)
phase_df=pd.DataFrame(phase_records)
df.to_csv(OUT_CSV,index=False,encoding='utf-8-sig')
phase_df.to_csv(OUT_SUMMARY,index=False,encoding='utf-8-sig')
print('動画:',OUT_MP4,f'({frame_count/FPS:.2f}秒 / {frame_count}フレーム)')
print('フレームCSV:',OUT_CSV,len(df),'行')
print('動作段階別CSV:',OUT_SUMMARY,len(phase_df),'行')
print('AI推論CSV:',OUT_INFERENCE,len(INFERENCE_RESULTS),'行')
print(phase_df[['phase','end_time_s','end_target_xy_error_m','end_sample_support_clearance_m',
                'end_both_pads_touch','end_target_support_contact_count']].to_string(index=False))
display(Video(OUT_MP4,embed=True,width=960))

## セル7：物理動作・AI棄却・HOMEのログ検証

第6回の把持・搬送・配置の判定基準に加え、REJECT時の指令保持とHOMEへの復帰を数値確認する。成功は**今回の選定例文・単一シミュレーション条件に限る**。

In [ ]:
# セル6：Pick & Place 一次評価（工業的性能・安全性を保証しない）
import matplotlib.pyplot as plt
transfer=df[df.phase.isin(['TRANSFER','HOLD_AT_TARGET'])]
final=df[df.phase=='FINAL_HOLD']
retract=df[df.phase.isin(['RETRACT','FINAL_HOLD'])]
close=df[df.phase.isin(['CLOSE','GRASP_HOLD'])]
if len(final)==0 or len(transfer)==0:
    raise RuntimeError('必要な動作段階がありません。セル5を再実行してください。')

close_contact=bool(close.both_pads_touch.any())
transport_both_rate=float(transfer.both_pads_touch.mean())
transport_max_dist=float(transfer.sample_to_pinch_distance_m.max())
last=df.iloc[-1]
final_xy_err=float(last.sample_to_target_xy_error_m)
final_clearance=float(last.sample_clearance_above_support_m)
final_tilt_alignment=float(last.sample_vertical_alignment)
final_source_err=float(last.sample_to_source_xy_error_m)
final_z_change=float(final.sample_z_m.max()-final.sample_z_m.min())
# 退避時に円柱を一緒に持ち上げていないかを独立に確認
retract_object_lift=float(retract.sample_z_m.max()-retract.sample_z_m.min())
retract_pinch_up=float(retract.pinch_z_m.iloc[-1]-retract.pinch_z_m.iloc[0])
contact_target_any=bool((df[df.phase.isin(['SETTLE_AT_TARGET','RELEASE','WAIT_AFTER_RELEASE','RETRACT','FINAL_HOLD'])].target_pedestal_contact_count>0).any())

checks={
    '閉じた際の両パッド接触':close_contact,
    '搬送フェーズの両パッド接触率 >= 0.75':transport_both_rate>=.75,
    '搬送中の試料と把持点の最大距離 < 45 mm':transport_max_dist<.045,
    '最終XY位置誤差 < 20 mm':final_xy_err<.020,
    '最終試料底面と台上面のずれ < 10 mm':abs(final_clearance)<.010,
    '試料が鉛直に近い（軸一致度 > 0.94）':final_tilt_alignment>.94,
    '試料が出発台から離れている（> 60 mm）':final_source_err>.060,
    '退避後の最終保持段階で高さ変動 < 5 mm':final_z_change<.005,
    '退避中の物体高さ変動 < 15 mm':retract_object_lift<.015,
    'グリッパーが退避した（> 40 mm）':retract_pinch_up>.040,
}
print('===== 第6回：搬送・配置の一次評価 =====')
for name,ok in checks.items(): print(('PASS' if ok else 'FAIL'),name)
print(f'搬送中の両パッド同時接触率: {transport_both_rate:.3f}')
print(f'搬送中の試料-把持点最大距離: {transport_max_dist*1000:.1f} mm')
print(f'搬送先台中心からの最終XY誤差: {final_xy_err*1000:.1f} mm')
print(f'最終試料底面と搬送先台上面の高さ差: {final_clearance*1000:.1f} mm')
print(f'搬送先台との接触を検出したか: {contact_target_any}')
print(f'退避中の試料高さ変動: {retract_object_lift*1000:.1f} mm')
print(f'グリッパー退避量: {retract_pinch_up*1000:.1f} mm')
preliminary_success=all(checks.values())
print('一次判定:', 'すべての基準に適合（追加の再現性検証が必要）' if preliminary_success else '基準未達（CSVと動画から原因分析が必要）')
if not contact_target_any:
    print('注意: 搬送先台とのMuJoCo接触は直接検出されませんでした。高さ・XY位置と合わせて判定してください。')

fig,axes=plt.subplots(3,1,figsize=(11,10),sharex=True)
axes[0].plot(df.time_s,df.sample_x_m,label='sample x')
axes[0].plot(df.time_s,df.sample_y_m,label='sample y')
axes[0].axhline(float(DESTINATION_XY[0]),ls='--',color='gray',alpha=.6,label='target x')
axes[0].axhline(float(DESTINATION_XY[1]),ls=':',color='gray',alpha=.6,label='target y')
axes[0].set_ylabel('world XY [m]');axes[0].legend();axes[0].grid(alpha=.25)
axes[1].plot(df.time_s,df.sample_z_m,label='sample z')
axes[1].plot(df.time_s,df.pinch_z_m,label='gripper z')
axes[1].axhline(SUPPORT_TOP_Z+HALF_HEIGHT_M,color='gray',ls='--',label='pedestal sample z')
axes[1].set_ylabel('height [m]');axes[1].legend();axes[1].grid(alpha=.25)
axes[2].plot(df.time_s,df.left_pad_normal_force_N,label='left normal [N]')
axes[2].plot(df.time_s,df.right_pad_normal_force_N,label='right normal [N]')
axes[2].set_ylabel('contact force [N]');axes[2].set_xlabel('simulated time [s]')
axes[2].legend();axes[2].grid(alpha=.25)
plt.tight_layout();plt.show()
# AI固有の判定を追加：REJECTに対して関節目標・グリッパー目標が変化しないこと
rejected=df[df.phase=='REJECT_HOLD']
home=df[df.phase=='HOME']
assert len(rejected)>1 and len(home)>1, 'REJECTまたはHOMEのフェーズログが不足しています'
reject_grip_held=bool(rejected.gripper_ctrl_0_255.nunique()==1)
# 最初のフェーズだけを保持とみなすのではなく、REJECT中のアーム関節目標が変わらない構成であることはセル6に明示。
home_q_error=float(home.max_arm_tracking_error_rad.iloc[-1])
no_new_command=bool(reject_grip_held)
print('===== 第7回：AI推論と制御の統合評価 =====')
for result in INFERENCE_RESULTS:
    print(f"{result['task_id']:>18} | score={result['score']:.4f} margin={result['margin']:.4f} | {result['instruction']}")
print('REJECT中のグリッパー制御値の保持:', no_new_command)
print('HOMEフェーズ終了時のアーム追従誤差 [rad]:', round(home_q_error,5))
print('Pick & Place一次判定:', preliminary_success)
print('研究上の限界: 日本語指示から軌道・把持位置をAIが生成したわけではない。')

## セル8：動画・4種類のログをWindowsへ保存

結果を確認後、07ノートブック（Colabのファイル→ダウンロード→`.ipynb`）とCSVをGitHubに追加する。既存01～06ファイルは上書きしない。**28秒動画は講座提出用の1～3分動画を置き換えない**。

In [ ]:
from google.colab import files
for f in [OUT_MP4, OUT_CSV, OUT_SUMMARY, OUT_INFERENCE]:
    files.download(f)

## 次の研究課題

1. 指示を変えた独立評価セットを作り、採択率と不適切作動率・誤棄却率を測定する。
2. 不明確な指示をREJECTした後の**実機相当の停止条件**を別途設計する（現在は新しい目標を与えない制御のみ）。
3. 円柱の質量・摩擦・初期位置を変えた反復試験と、接触剛性・接触力の妥当性確認。
4. 対象物位置の認識と逆方向の作業追加。段階的にROS 2／実機へ拡張する。
5. 動作系列の衝突判定、軌道計画、実機安全性の検証。

**まだ開発中**：今回は選定済みの指示から、定義済みPick & Place系列を実行する研究用試作である。